In [1]:
from pathlib import Path
import pyreadstat
import pandas as pd

# Resolve project root dynamically for cross-machine reproducibility
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = PROJECT_ROOT / "data"

# Point strictly to the official PISA 2022 (CY08) SAS files
school_file_path = DATA_DIR / "CY09_MS_SCH_PUF" / "SCH_QQQ_SAS" / "CY08MSP_SCH_QQQ.SAS7BDAT"
student_file_path = DATA_DIR / "STU_QQQ_SAS" / "CY08MSP_STU_QQQ.SAS7BDAT"

# Load metadata and lightweight preview to verify PISA 2022 cycle (CY08)
df_school_preview, school_meta = pyreadstat.read_sas7bdat(
    school_file_path, 
    usecols=['CNT', 'CNTSCHID', 'STRATIO', 'PROATCE', 'EDUSHORT', 'CLSIZE', 'SCHLTYPE']
)

print(f"PISA 2022 School dataset shape: {df_school_preview.shape[0]} schools, {df_school_preview.shape[1]} variables.")
df_school_preview.head(5)

PISA 2022 School dataset shape: 21629 schools, 7 variables.


,CNT,CNTSCHID,SCHLTYPE,PROATCE,STRATIO,EDUSHORT,CLSIZE
0,ALB,800001.0,3.0,1.0000,16.9351,1.2048,33.0
1,ALB,800002.0,3.0,1.0000,11.4375,2.9595,13.0
2,ALB,800003.0,1.0,1.0000,7.1176,0.6299,13.0
3,ALB,800004.0,3.0,0.9449,16.9134,1.6863,28.0
4,ALB,800005.0,3.0,0.9630,11.7407,1.2478,18.0


In [2]:
# Load minimal student identifiers and weights to inspect country sample distributions
df_stu_meta, _ = pyreadstat.read_sas7bdat(
    student_file_path,
    usecols=['CNT', 'CNTSCHID', 'ESCS', 'W_FSTUWT', 'PV1MATH']
)

# Check if Ukraine is coded as 'UKR' or 'QUR' (Ukrainian Regions in PISA 2022)
ukr_codes = [c for c in df_stu_meta['CNT'].unique() if 'U' in str(c) or 'Q' in str(c)]
print("Detected codes matching Ukraine / Regions:", [c for c in ukr_codes if c in ['UKR', 'QUR']])

# Inspect sample sizes and total population weights across target European countries
candidate_countries = ['DEU', 'FRA', 'POL', 'EST', 'ITA', 'ESP', 'AUT', 'BEL', 'CZE', 'FIN', 'NLD', 'PRT', 'SWE', 'QUR']
cohort_summary = (
    df_stu_meta[df_stu_meta['CNT'].isin(candidate_countries)]
    .groupby('CNT')
    .agg(
        raw_students=('CNTSCHID', 'count'),
        unique_schools=('CNTSCHID', 'nunique'),
        missing_escs=('ESCS', lambda x: x.isna().sum()),
        sum_weights=('W_FSTUWT', 'sum')
    )
)
cohort_summary['raw_sample_share_%'] = (100 * cohort_summary['raw_students'] / cohort_summary['raw_students'].sum()).round(2)
cohort_summary.sort_values(by='raw_students', ascending=False)

Detected codes matching Ukraine / Regions: ['QUR']


,raw_students,unique_schools,missing_escs,sum_weights,raw_sample_share_%
CNT,,,,,
ESP,30800,966,1152,459029.42343,25.34
ITA,10552,344,101,496262.80962,8.68
FIN,10239,241,308,58954.73953,8.42
CZE,8460,430,159,100265.96644,6.96
BEL,8286,285,286,128641.97592,6.82
PRT,6793,224,119,96606.89403,5.59
FRA,6770,282,198,781285.70627,5.57
EST,6392,196,101,13344.95332,5.26
AUT,6151,302,170,76153.19564,5.06


In [3]:
# Summary diagnostics justifying pipeline architecture decisions
print("Exploration Takeaways for Main Pipeline:")
print("1. Cycle verified: Using official PISA 2022 (CY08MSP) data.")
print("2. Country coverage: Ukraine is coded as 'QUR' (18 surveyed regions). Expanding European cohort to 14 systems.")
print("3. Sample balance: Spain (ESP) accounts for a disproportionate share of raw observations; Senate Weights (SENWT) are required.")
print("4. Missingness: ESCS has systematic missingness; we will add binary imputation flags before country-median imputation.")

Exploration Takeaways for Main Pipeline:
1. Cycle verified: Using official PISA 2022 (CY08MSP) data.
2. Country coverage: Ukraine is coded as 'QUR' (18 surveyed regions). Expanding European cohort to 14 systems.
3. Sample balance: Spain (ESP) accounts for a disproportionate share of raw observations; Senate Weights (SENWT) are required.
4. Missingness: ESCS has systematic missingness; we will add binary imputation flags before country-median imputation.
